# Hey Banco — Datathon 2026
## EDA UC3: Cashback Perdido — Upselling Inteligente Hey Pro

**Objetivo:** Cuantificar el cashback que los usuarios sin Hey Pro están dejando de ganar para generar el dato duro que Havi usa en su mensaje proactivo: *"dejaste X pesos en la mesa el mes pasado"*.

| Métrica clave | Descripción |
|---------------|-------------|
| Cashback total perdido | Monto MXN que no-Pro dejaron de ganar en el período |
| Cashback perdido mensual | Promedio por usuario/mes para el mensaje de Havi |
| Segmentos A / B / C | Priorización de usuarios por potencial de conversión |
| Activación inmediata | % segmento A con nómina domiciliada ya activa |
| Señal conversacional | ¿Usuarios no-Pro preguntan sobre cashback/beneficios? |

---
**Regla de negocio:** `cashback_potencial = monto × 0.01` por compra completada con Hey Pro  
**Filtros base:** `es_hey_pro=False` · `tipo_operacion='compra'` · `estatus='completada'`

## 0. Setup y carga de datos

In [ ]:
import subprocess, sys
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 100)

# ── Paleta UC3 ──────────────────────────────────────────────────────────────
C = {
    'pro':      '#1D3557',   # azul oscuro = Hey Pro
    'nopro':    '#E63946',   # rojo = cashback perdido
    'seg_a':    '#E63946',   # segmento A (mayor potencial)
    'seg_b':    '#F4A261',   # segmento B
    'seg_c':    '#A8DADC',   # segmento C
    'nomina':   '#457B9D',   # nómina domiciliada
    'cat':      '#2A9D8F',   # categorías
    'convs':    '#E9C46A',   # conversaciones
}

BASE_TXN  = Path(r"Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path(r"Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")
print("Rutas:", BASE_TXN.resolve(), "|", BASE_CONV.resolve())

In [ ]:
df_clientes = pd.read_csv(
    BASE_TXN / "hey_clientes.csv",
    dtype={"user_id": str}
)

df_tx = pd.read_csv(
    BASE_TXN / "hey_transacciones.csv",
    dtype={"transaccion_id": str, "user_id": str, "producto_id": str},
    parse_dates=["fecha_hora"]
)

df_convs = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")
df_convs["date"] = pd.to_datetime(df_convs["date"], format="mixed")

# Quitar columna de metadato sintético si existe
for df in [df_tx]:
    if "es_dato_sintetico" in df.columns:
        df.drop(columns=["es_dato_sintetico"], inplace=True)

print(f"Clientes:       {len(df_clientes):>8,}")
print(f"Transacciones:  {len(df_tx):>8,}")
print(f"Conversaciones: {len(df_convs):>8,}")

## 1. Universo No-Pro y compras completadas

In [ ]:
# ── 1.1 Usuarios sin Hey Pro ─────────────────────────────────────────────────
no_pro_ids = df_clientes.loc[df_clientes["es_hey_pro"] == False, "user_id"]
pro_ids    = df_clientes.loc[df_clientes["es_hey_pro"] == True,  "user_id"]

n_nopro = len(no_pro_ids)
n_pro   = len(pro_ids)
total_cli = len(df_clientes)

print(f"Total clientes : {total_cli:,}")
print(f"  → Hey Pro    : {n_pro:,}  ({n_pro/total_cli:.1%})")
print(f"  → No-Pro     : {n_nopro:,}  ({n_nopro/total_cli:.1%})")

# ── 1.2 Compras completadas de usuarios No-Pro ───────────────────────────────
compras = df_tx[
    (df_tx["user_id"].isin(no_pro_ids)) &
    (df_tx["tipo_operacion"] == "compra") &
    (df_tx["estatus"] == "completada")
].copy()

compras["cashback_potencial"] = compras["monto"] * 0.01
compras["mes"] = compras["fecha_hora"].dt.to_period("M")

print(f"\nCompras completadas (no-Pro): {len(compras):,}")
print(f"Período: {compras['fecha_hora'].min().date()} → {compras['fecha_hora'].max().date()}")
print(f"Cashback potencial total: ${compras['cashback_potencial'].sum():>12,.2f} MXN")

## 2. Cashback perdido mensual por usuario

In [ ]:
# ── 2.1 Cashback por usuario × mes ──────────────────────────────────────────
cb_user_mes = (
    compras
    .groupby(["user_id", "mes"])["cashback_potencial"]
    .sum()
    .reset_index()
    .rename(columns={"cashback_potencial": "cashback_mes"})
)

# Cashback promedio mensual por usuario (sobre los meses en que tuvo compras)
cb_user_avg = (
    cb_user_mes
    .groupby("user_id")["cashback_mes"]
    .mean()
    .reset_index()
    .rename(columns={"cashback_mes": "cashback_perdido_mensual"})
)

# Usuarios no-Pro SIN ninguna compra en el período → cashback = 0
cb_full = (
    pd.DataFrame({"user_id": no_pro_ids})
    .merge(cb_user_avg, on="user_id", how="left")
)
cb_full["cashback_perdido_mensual"] = cb_full["cashback_perdido_mensual"].fillna(0)

print("Estadísticas de cashback_perdido_mensual (MXN):")
print(cb_full["cashback_perdido_mensual"].describe(percentiles=[.25,.5,.75,.90,.95,.99]).apply("${:,.2f}".format))

In [ ]:
# ── 2.2 Distribución visual del cashback mensual perdido ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cashback Mensual Perdido — Usuarios No-Pro", fontsize=14, fontweight="bold", y=1.01)

data = cb_full["cashback_perdido_mensual"]
data_clipped = data.clip(upper=data.quantile(0.99))  # clip outliers para visualización

# Histograma
ax1 = axes[0]
ax1.hist(data_clipped[data_clipped > 0], bins=50, color=C['nopro'], alpha=0.85, edgecolor='white', linewidth=0.3)
ax1.axvline(data.median(), color='navy', lw=1.5, ls='--', label=f"Mediana: ${data.median():,.0f}")
ax1.axvline(data.mean(),   color='black', lw=1.5, ls=':',  label=f"Media:   ${data.mean():,.0f}")
ax1.set_xlabel("Cashback mensual perdido (MXN)")
ax1.set_ylabel("Nº usuarios")
ax1.set_title("Histograma (p99 clipped)")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(fontsize=9)

# Boxplot por percentiles clave
ax2 = axes[1]
pcts = [10, 25, 50, 75, 90, 95, 99]
vals = [np.percentile(data, p) for p in pcts]
bars = ax2.barh([f"p{p}" for p in pcts], vals, color=C['nopro'], alpha=0.8)
for bar, val in zip(bars, vals):
    ax2.text(val + 2, bar.get_y() + bar.get_height()/2, f"${val:,.0f}", va='center', fontsize=9)
ax2.set_xlabel("Cashback mensual perdido (MXN)")
ax2.set_title("Percentiles")

plt.tight_layout()
plt.show()
print(f"Usuarios con cashback > $0/mes: {(data > 0).sum():,} / {len(data):,} ({(data > 0).mean():.1%})")

## 3. Segmentación A / B / C por potencial de cashback

In [ ]:
# ── 3.1 Definir segmentos ────────────────────────────────────────────────────
#   A: > $300/mes  → alta oportunidad
#   B: $100–$300   → oportunidad media
#   C: < $100      → bajo potencial

def asignar_segmento(x):
    if x > 300:   return "A"
    elif x >= 100: return "B"
    else:          return "C"

cb_full["segmento"] = cb_full["cashback_perdido_mensual"].apply(asignar_segmento)

seg_counts = cb_full["segmento"].value_counts().sort_index()
seg_pct    = seg_counts / len(cb_full) * 100
seg_total_cb = cb_full.groupby("segmento")["cashback_perdido_mensual"].sum()
seg_avg_cb   = cb_full.groupby("segmento")["cashback_perdido_mensual"].mean()

resumen_seg = pd.DataFrame({
    "Usuarios": seg_counts,
    "% del total": seg_pct.map("{:.1f}%".format),
    "Cashback perdido/mes promedio": seg_avg_cb.map("${:,.2f}".format),
    "Cashback total mensual": seg_total_cb.map("${:,.0f}".format),
})
print("=" * 70)
print("SEGMENTACIÓN DE USUARIOS NO-PRO POR CASHBACK MENSUAL POTENCIAL")
print("=" * 70)
print(resumen_seg.to_string())
print(f"\n{'TOTAL':>10}: {seg_counts.sum():>6,} usuarios  |  ${seg_total_cb.sum():>10,.0f} MXN/mes perdido")

In [ ]:
# ── 3.2 Visualización de segmentos ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Segmentación Cashback Perdido — Usuarios No-Pro", fontsize=14, fontweight="bold")

seg_colors = {"A": C["seg_a"], "B": C["seg_b"], "C": C["seg_c"]}
labels_seg = ["A\n(>$300/mes)", "B\n($100–$300)", "C\n(<$100)"]
seg_order  = ["A", "B", "C"]
colors_ord = [seg_colors[s] for s in seg_order]

# Donut — nº de usuarios por segmento
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    [seg_counts[s] for s in seg_order],
    labels=labels_seg, autopct="%1.1f%%",
    colors=colors_ord, startangle=90,
    wedgeprops={"width": 0.55, "edgecolor": "white"},
    textprops={"fontsize": 10}
)
ax1.set_title(f"% Usuarios\n(total: {seg_counts.sum():,})", fontsize=11)

# Barras — cashback total mensual
ax2 = axes[1]
bar_vals = [seg_total_cb.get(s, 0) for s in seg_order]
bars = ax2.bar(labels_seg, bar_vals, color=colors_ord, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, bar_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 500, f"${val:,.0f}",
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_ylabel("MXN / mes")
ax2.set_title("Cashback total perdido / mes")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

# Barras — cashback promedio por usuario
ax3 = axes[2]
avg_vals = [seg_avg_cb.get(s, 0) for s in seg_order]
bars3 = ax3.bar(labels_seg, avg_vals, color=colors_ord, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars3, avg_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 1, f"${val:,.0f}",
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax3.set_ylabel("MXN / mes")
ax3.set_title("Cashback promedio por usuario")

plt.tight_layout()
plt.show()

## 4. Top categorías donde más cashback se pierde

In [ ]:
# ── 4.1 Cashback por categoría MCC ──────────────────────────────────────────
cb_por_cat = (
    compras
    .groupby("categoria_mcc")
    .agg(
        cashback_total   =("cashback_potencial", "sum"),
        n_transacciones  =("transaccion_id",     "count"),
        n_usuarios       =("user_id",             "nunique"),
        monto_total      =("monto",               "sum"),
    )
    .sort_values("cashback_total", ascending=False)
    .reset_index()
)

cb_por_cat["% del cashback"] = cb_por_cat["cashback_total"] / cb_por_cat["cashback_total"].sum() * 100
cb_por_cat["cashback_x_usuario"] = cb_por_cat["cashback_total"] / cb_por_cat["n_usuarios"]

TOP_N = 15
top_cat = cb_por_cat.head(TOP_N).copy()

# Destacar categorías clave UC3
KEY_CATS = {"restaurante", "gasolina", "supermercado", "servicios_digitales", "viajes"}

print(f"Top {TOP_N} categorías por cashback perdido:")
display_cols = ["categoria_mcc", "cashback_total", "% del cashback", "n_transacciones", "n_usuarios", "cashback_x_usuario"]
fmt_df = top_cat[display_cols].copy()
fmt_df["cashback_total"]     = fmt_df["cashback_total"].map("${:,.2f}".format)
fmt_df["% del cashback"]     = fmt_df["% del cashback"].map("{:.1f}%".format)
fmt_df["cashback_x_usuario"] = fmt_df["cashback_x_usuario"].map("${:,.2f}".format)
print(fmt_df.to_string(index=False))

In [ ]:
# ── 4.2 Gráfica de categorías ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Top Categorías — Cashback Mensual Perdido (No-Pro)", fontsize=13, fontweight="bold")

# Barras horizontales — cashback total
ax1 = axes[0]
cats   = top_cat["categoria_mcc"].tolist()[::-1]
cbs    = top_cat["cashback_total"].tolist()[::-1]
bar_colors = [C["seg_a"] if c in KEY_CATS else C["cat"] for c in cats]
ax1.barh(cats, cbs, color=bar_colors, edgecolor="white", linewidth=0.4)
ax1.set_xlabel("Cashback perdido (MXN)")
ax1.set_title("Por cashback total")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
key_patch  = mpatches.Patch(color=C["seg_a"], label="Categorías clave UC3")
other_patch = mpatches.Patch(color=C["cat"],  label="Otras")
ax1.legend(handles=[key_patch, other_patch], fontsize=9)

# Barras horizontales — cashback por usuario
ax2 = axes[1]
cbu = top_cat["cashback_x_usuario"].tolist()[::-1]
ax2.barh(cats, cbu, color=bar_colors, edgecolor="white", linewidth=0.4)
ax2.set_xlabel("Cashback perdido por usuario (MXN)")
ax2.set_title("Por cashback / usuario")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

## 5. Cruce con nómina domiciliada — activación inmediata

In [ ]:
# ── 5.1 Enriquecer con perfil del cliente ────────────────────────────────────
cb_perfil = cb_full.merge(
    df_clientes[["user_id", "nomina_domiciliada", "ingreso_mensual_mxn",
                 "score_buro", "edad", "ocupacion", "satisfaccion_1_10",
                 "dias_desde_ultimo_login", "num_productos_activos"]],
    on="user_id", how="left"
)

# ── 5.2 Segmento A × nómina domiciliada (activación inmediata) ───────────────
seg_a = cb_perfil[cb_perfil["segmento"] == "A"]

nomina_en_seg_a = seg_a["nomina_domiciliada"].sum()
pct_nomina_seg_a = nomina_en_seg_a / len(seg_a)

print("=" * 60)
print("SEGMENTO A — Usuarios con >$300 cashback/mes potencial")
print("=" * 60)
print(f"Total segmento A    : {len(seg_a):,} usuarios")
print(f"Con nómina dom.     : {nomina_en_seg_a:,} ({pct_nomina_seg_a:.1%}) → activación INMEDIATA")
print(f"Sin nómina dom.     : {len(seg_a) - nomina_en_seg_a:,} ({1-pct_nomina_seg_a:.1%}) → upsell paso 2")
print()
print("Perfil Segmento A:")
print(seg_a[["cashback_perdido_mensual", "ingreso_mensual_mxn", "score_buro", "edad"]]
      .describe(percentiles=[.25,.5,.75])
      .round(1)
      .to_string())

In [ ]:
# ── 5.3 Visualización: nómina × segmento ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Nómina Domiciliada × Segmento UC3", fontsize=13, fontweight="bold")

# Barras apiladas: con / sin nómina por segmento
ax1 = axes[0]
con_nomina    = cb_perfil.groupby("segmento")["nomina_domiciliada"].sum().reindex(seg_order)
sin_nomina    = cb_perfil.groupby("segmento")["nomina_domiciliada"].apply(lambda x: (~x).sum()).reindex(seg_order)
total_seg     = con_nomina + sin_nomina

x = np.arange(len(seg_order))
w = 0.5
b1 = ax1.bar(x, con_nomina.values, w, label="Con nómina (activación inmediata)", color=C["nomina"])
b2 = ax1.bar(x, sin_nomina.values, w, bottom=con_nomina.values, label="Sin nómina", color="#CBD4E1")
for i, (cn, sn, tot) in enumerate(zip(con_nomina, sin_nomina, total_seg)):
    ax1.text(i, tot + 10, f"{cn/tot:.0%} nómina", ha='center', fontsize=9, fontweight='bold', color=C["nomina"])
ax1.set_xticks(x)
ax1.set_xticklabels(["A\n(>$300)", "B\n($100-$300)", "C\n(<$100)"])
ax1.set_ylabel("Nº usuarios")
ax1.set_title("Usuarios con/sin nómina por segmento")
ax1.legend(fontsize=9)

# Boxplot: cashback por segmento + nómina
ax2 = axes[1]
seg_a_nom  = seg_a.loc[seg_a["nomina_domiciliada"] == True,  "cashback_perdido_mensual"]
seg_a_nonom = seg_a.loc[seg_a["nomina_domiciliada"] == False, "cashback_perdido_mensual"]
bplot = ax2.boxplot([seg_a_nom, seg_a_nonom],
                    labels=["Seg A\nCon nómina", "Seg A\nSin nómina"],
                    patch_artist=True,
                    medianprops={"color": "white", "linewidth": 2})
for patch, color in zip(bplot["boxes"], [C["nomina"], "#CBD4E1"]):
    patch.set_facecolor(color)
ax2.set_ylabel("Cashback mensual perdido (MXN)")
ax2.set_title("Distribución cashback — Segmento A")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

## 6. Señal conversacional — Usuarios no-Pro preguntando sobre cashback / beneficios

In [ ]:
# ── 6.1 Palabras clave de interés en cashback / Hey Pro ──────────────────────
KEYWORDS_CB = [
    r"cashback", r"cash back", r"beneficio", r"descuento", r"recompensa",
    r"hey pro", r"heypro", r"suscripci.n", r"contrat",
    r"\d+%", r"porcentaje", r"devoluc", r"ahorro", r"cu.nto gano",
    r"qu. incluye", r"para qu. sirve", r"ventaja", r"plan",
]

pattern_cb = "|".join(KEYWORDS_CB)

# Marcar turnos con mención de cashback/beneficios en el input del usuario
df_convs["menciona_cb"] = (
    df_convs["input"]
    .fillna("")
    .str.lower()
    .str.contains(pattern_cb, regex=True, na=False)
)

# Enriquecer con perfil del cliente
convs_perfil = df_convs.merge(
    df_clientes[["user_id", "es_hey_pro", "nomina_domiciliada", "segmento_propio"] 
                 if "segmento_propio" in df_clientes.columns
                 else ["user_id", "es_hey_pro", "nomina_domiciliada"]],
    on="user_id", how="left"
)

# Segmento UC3 para cada user_id en conversaciones
convs_perfil = convs_perfil.merge(
    cb_full[["user_id", "cashback_perdido_mensual", "segmento"]],
    on="user_id", how="left"
)

# ── 6.2 Usuarios no-Pro que mencionaron cashback/beneficios ──────────────────
nopro_convs = convs_perfil[convs_perfil["es_hey_pro"] == False]
nopro_cb_mention = nopro_convs[nopro_convs["menciona_cb"] == True]

n_unique_nopro   = nopro_convs["user_id"].nunique()
n_unique_mention = nopro_cb_mention["user_id"].nunique()

print(f"Usuarios no-Pro con conversaciones en Havi  : {n_unique_nopro:,}")
print(f"Usuarios no-Pro con mención cashback/beneficios: {n_unique_mention:,}  ({n_unique_mention/n_unique_nopro:.1%})")
print()
print("Por segmento UC3:")
seg_cb_mention = (
    nopro_cb_mention
    .drop_duplicates("user_id")
    .groupby("segmento")
    .size()
    .reindex(["A","B","C"])
    .fillna(0)
    .astype(int)
)
seg_total_users = cb_full["segmento"].value_counts().reindex(["A","B","C"]).fillna(0)
for s in ["A","B","C"]:
    n_m = seg_cb_mention.get(s, 0)
    n_t = seg_total_users.get(s, 0)
    pct = n_m/n_t if n_t > 0 else 0
    print(f"  Segmento {s}: {n_m:>4,} de {n_t:>5,} usuarios preguntaron  ({pct:.1%})")

In [ ]:
# ── 6.3 Ejemplos representativos de mensajes (no-Pro que preguntan sobre cashback) ──
import unicodedata

def clean_text(t):
    if not isinstance(t, str): return ""
    return unicodedata.normalize("NFKD", t).encode("ascii", "replace").decode("ascii").replace("?", "")

samples = (
    nopro_cb_mention
    .merge(cb_full[["user_id","cashback_perdido_mensual","segmento"]], on="user_id", how="left")
    .sort_values("cashback_perdido_mensual", ascending=False)
    .drop_duplicates(subset=["conv_id"])
    .head(10)[["user_id", "segmento", "cashback_perdido_mensual", "input"]]
)
samples["input_clean"] = samples["input"].apply(clean_text)
samples["cashback_perdido_mensual"] = samples["cashback_perdido_mensual"].map("${:,.2f}".format)

print("Top 10 conversaciones — Usuarios no-Pro de mayor cashback potencial:\n")
for _, row in samples.iterrows():
    print(f"  [{row['segmento']}] {row['user_id']} | perdido/mes: {row['cashback_perdido_mensual']}")
    print(f"  → \"{row['input_clean'][:120]}\"")
    print()

In [ ]:
# ── 6.4 Visualización: señal conversacional por segmento ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Señal Conversacional UC3 — Usuarios No-Pro sobre Cashback/Hey Pro", fontsize=12, fontweight="bold")

# Barras: % que preguntó por segmento
ax1 = axes[0]
pct_mention = [seg_cb_mention.get(s, 0) / seg_total_users.get(s, 1) * 100 for s in ["A","B","C"]]
bars = ax1.bar(["A\n(>$300)","B\n($100-$300)","C\n(<$100)"], pct_mention,
               color=[C["seg_a"], C["seg_b"], C["seg_c"]], edgecolor="white")
for bar, val in zip(bars, pct_mention):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.1, f"{val:.1f}%",
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_ylabel("% de usuarios que mencionaron cashback/beneficios")
ax1.set_title("% con señal de intención por segmento")

# Distribución mensual de conversaciones sobre cashback (no-Pro)
ax2 = axes[1]
nopro_cb_mention["mes_conv"] = nopro_cb_mention["date"].dt.to_period("M")
mensual = nopro_cb_mention.groupby("mes_conv").size()
mensual.index = mensual.index.astype(str)
ax2.bar(mensual.index, mensual.values, color=C["convs"], edgecolor="white")
ax2.set_xlabel("Mes")
ax2.set_ylabel("Nº de turnos")
ax2.set_title("Evolución mensual — preguntas cashback/Hey Pro")
plt.xticks(rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.show()

## 7. Resumen ejecutivo — Criterios de aceptación UC3

In [ ]:
# ── 7.1 Cálculo de KPIs finales ───────────────────────────────────────────────
total_cashback_perdido = compras["cashback_potencial"].sum()
meses_periodo = compras["mes"].nunique()
cashback_perdido_mensual_universo = total_cashback_perdido / meses_periodo

seg_dist = cb_full["segmento"].value_counts() / len(cb_full)

seg_a_nomina_pct   = (seg_a["nomina_domiciliada"] == True).mean()

convs_intention_pct = n_unique_mention / n_unique_nopro

# Mensaje de Havi para el usuario mediana del segmento A
mediana_seg_a = seg_a["cashback_perdido_mensual"].median()
mediana_seg_b = cb_perfil.loc[cb_perfil["segmento"]=="B", "cashback_perdido_mensual"].median()

print("=" * 70)
print("CRITERIOS DE ACEPTACIÓN — UC3 CASHBACK PERDIDO")
print("=" * 70)
print()
print(f"[✅] Cashback total perdido (período completo)  : ${total_cashback_perdido:>12,.2f} MXN")
print(f"[✅] Cashback perdido mensual (universo no-Pro) : ${cashback_perdido_mensual_universo:>12,.2f} MXN/mes")
print(f"[✅] Período analizado                         : {meses_periodo} meses")
print()
print(f"[✅] % Segmento A (>$300/mes)    : {seg_dist.get('A',0):.1%}  → {int(len(cb_full)*seg_dist.get('A',0)):,} usuarios")
print(f"[✅] % Segmento B ($100-$300/mes): {seg_dist.get('B',0):.1%}  → {int(len(cb_full)*seg_dist.get('B',0)):,} usuarios")
print(f"[✅] % Segmento C (<$100/mes)    : {seg_dist.get('C',0):.1%}  → {int(len(cb_full)*seg_dist.get('C',0)):,} usuarios")
print()
print(f"[✅] % Seg A con nómina domiciliada (activación inmediata): {seg_a_nomina_pct:.1%}")
print()
print(f"[✅] % usuarios no-Pro con señal conversacional: {convs_intention_pct:.1%}")
print()
print("-" * 70)
print("MENSAJES DE HAVI SUGERIDOS:")
print(f"  Seg A median: 'El mes pasado dejaste ${mediana_seg_a:,.0f} en la mesa.")
print(f"               Con Hey Pro ya los tendrías de vuelta.'")
print(f"  Seg B median: 'El mes pasado dejaste ${mediana_seg_b:,.0f} en cashback.")
print(f"               ¿Activamos Hey Pro ahora?'")
print("-" * 70)

In [ ]:
# ── 7.2 Cuadro de mando final ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("UC3 — Cashback Perdido: Dashboard Ejecutivo No-Pro", fontsize=14, fontweight="bold", y=1.01)

# ① Cashback total perdido vs. generado Hey Pro
ax = axes[0,0]
cb_pro_total = df_tx[
    (df_tx["user_id"].isin(pro_ids)) &
    (df_tx["tipo_operacion"] == "compra") &
    (df_tx["estatus"] == "completada")
]["cashback_generado"].sum()
bars = ax.bar(["No-Pro\n(perdido)","Hey Pro\n(generado)"],
              [total_cashback_perdido, cb_pro_total],
              color=[C["nopro"], C["pro"]], edgecolor="white", width=0.55)
for bar, val in zip(bars, [total_cashback_perdido, cb_pro_total]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5000, f"${val/1e6:.2f}M",
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel("MXN")
ax.set_title("Cashback total (período completo)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))

# ② Distribución segmentos
ax = axes[0,1]
seg_c_vals = [seg_counts.get(s,0) for s in ["A","B","C"]]
wedges, texts, autotexts = ax.pie(
    seg_c_vals, labels=[f"A\n({seg_counts.get('A',0):,})",f"B\n({seg_counts.get('B',0):,})",f"C\n({seg_counts.get('C',0):,})"],
    autopct="%1.1f%%", colors=[C["seg_a"],C["seg_b"],C["seg_c"]],
    startangle=90, wedgeprops={"width":0.55,"edgecolor":"white"}
)
ax.set_title("Segmentos A/B/C (usuarios)")

# ③ Nómina × segmento A
ax = axes[0,2]
labels_nom = ["Con nómina\n(activ. inmediata)", "Sin nómina"]
vals_nom   = [seg_a["nomina_domiciliada"].sum(), (~seg_a["nomina_domiciliada"]).sum()]
wedges2, _, auto2 = ax.pie(
    vals_nom, labels=labels_nom, autopct="%1.1f%%",
    colors=[C["nomina"],"#CBD4E1"], startangle=90,
    wedgeprops={"width":0.55,"edgecolor":"white"}
)
ax.set_title(f"Seg A ({len(seg_a):,} usuarios)\nnómina domiciliada")

# ④ Top 8 categorías
ax = axes[1,0]
top8 = cb_por_cat.head(8)
bar_c = [C["seg_a"] if c in KEY_CATS else C["cat"] for c in top8["categoria_mcc"]]
ax.barh(top8["categoria_mcc"][::-1], top8["cashback_total"][::-1], color=bar_c[::-1], edgecolor="white")
ax.set_xlabel("Cashback perdido (MXN)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.set_title("Top categorías")

# ⑤ Distribución cashback mensual
ax = axes[1,1]
cb_positivo = cb_full.loc[cb_full["cashback_perdido_mensual"] > 0, "cashback_perdido_mensual"]
ax.hist(cb_positivo.clip(upper=cb_positivo.quantile(0.98)), bins=40,
        color=C["nopro"], alpha=0.8, edgecolor="white", linewidth=0.3)
ax.axvline(100, color='orange', lw=1.5, ls='--', label="Umbral B ($100)")
ax.axvline(300, color='navy',   lw=1.5, ls='--', label="Umbral A ($300)")
ax.set_xlabel("Cashback mensual perdido (MXN)")
ax.set_ylabel("Usuarios")
ax.set_title("Distribución cashback mensual")
ax.legend(fontsize=8)

# ⑥ Señal conversacional
ax = axes[1,2]
pct_m = [seg_cb_mention.get(s,0) / seg_total_users.get(s,1) * 100 for s in ["A","B","C"]]
bars3 = ax.bar(["A","B","C"], pct_m, color=[C["seg_a"],C["seg_b"],C["seg_c"]], edgecolor="white")
for bar, val in zip(bars3, pct_m):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, f"{val:.1f}%",
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel("%")
ax.set_title("% con intención cashback\n(señal conversacional)")

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Exportar tabla de usuarios no-Pro con segmento para UC3 model ────────
import json
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# CSV con todos los usuarios no-Pro y sus métricas
export_df = cb_perfil[[
    "user_id", "cashback_perdido_mensual", "segmento",
    "nomina_domiciliada", "ingreso_mensual_mxn", "score_buro",
    "edad", "ocupacion", "satisfaccion_1_10", "dias_desde_ultimo_login",
    "num_productos_activos"
]].copy()

export_path_csv = output_dir / "uc3_nopro_cashback_segmentos.csv"
export_df.to_csv(export_path_csv, index=False)
print(f"✅ CSV exportado: {export_path_csv}  ({len(export_df):,} usuarios)")

# JSON con KPIs ejecutivos
kpis = {
    "descripcion": "UC3 — Cashback Perdido No-Pro — EDA resultados",
    "periodo_meses": int(meses_periodo),
    "universo_nopro": int(len(cb_full)),
    "cashback_total_perdido_mxn": round(float(total_cashback_perdido), 2),
    "cashback_mensual_universo_mxn": round(float(cashback_perdido_mensual_universo), 2),
    "segmentos": {
        "A": {
            "descripcion": ">$300/mes",
            "n_usuarios": int(seg_counts.get("A",0)),
            "pct_universo": round(float(seg_dist.get("A",0)), 4),
            "pct_con_nomina": round(float(seg_a_nomina_pct), 4),
            "cashback_mediana": round(float(mediana_seg_a), 2),
        },
        "B": {
            "descripcion": "$100–$300/mes",
            "n_usuarios": int(seg_counts.get("B",0)),
            "pct_universo": round(float(seg_dist.get("B",0)), 4),
            "cashback_mediana": round(float(mediana_seg_b), 2),
        },
        "C": {
            "descripcion": "<$100/mes",
            "n_usuarios": int(seg_counts.get("C",0)),
            "pct_universo": round(float(seg_dist.get("C",0)), 4),
        }
    },
    "senal_conversacional_pct": round(float(convs_intention_pct), 4),
    "mensaje_havi_seg_a": f"El mes pasado dejaste ${mediana_seg_a:,.0f} en la mesa. Con Hey Pro ya los tendrías de vuelta.",
    "mensaje_havi_seg_b": f"El mes pasado dejaste ${mediana_seg_b:,.0f} en cashback. ¿Activamos Hey Pro ahora?",
}

export_path_json = output_dir / "uc3_cashback_kpis.json"
with open(export_path_json, "w", encoding="utf-8") as f:
    json.dump(kpis, f, ensure_ascii=False, indent=2)
print(f"✅ JSON exportado: {export_path_json}")

print("\n--- RESUMEN FINAL ---")
print(json.dumps(kpis, ensure_ascii=False, indent=2))